# Preprocessing Pipeline

## 1. What Is a Preprocessing Pipeline?

A preprocessing pipeline is a sequence of data preparation steps organized into a single workflow before or during Machine Learning model training.

Raw data may contain missing values, numerical features with different scales, and categorical values that models cannot directly process.

A pipeline allows us to combine preprocessing operations and a Machine Learning model into one organized process.

**Example:** Suppose we want to predict customer churn. Our dataset contains:

* Missing values in monthly charges.
* Numerical features such as tenure and monthly charges.
* Categorical features such as contract type.
* A target column indicating whether the customer churned.

A preprocessing pipeline can fill missing values, scale numerical features, encode categorical features, and train a classification model.

## 2. Why Do We Need a Preprocessing Pipeline?

Pipelines help us:

* Organize multiple preprocessing steps.
* Apply the same transformations consistently.
* Reduce repetitive code.
* Prevent common forms of data leakage.
* Simplify model training and prediction.
* Support cross-validation and hyperparameter tuning.
* Make Machine Learning workflows easier to maintain.

Without a pipeline, we may have to write and manage each preprocessing step separately, increasing the chance of mistakes.

## 3. What Is a Machine Learning Workflow?

A typical workflow is:

1. Load the dataset.
2. Inspect and understand the data.
3. Separate input features and the target.
4. Split the data into training and testing sets.
5. Preprocess the input features.
6. Train the Machine Learning model.
7. Evaluate the model on the test set.
8. Use the fitted workflow to predict on new data.

A pipeline can combine steps 5 and 6 so they are executed consistently.

## 4. Important Pipeline Components

### A. Missing Value Imputation

Imputation fills missing values using a selected strategy.

Examples:

* Mean imputation: replaces missing numerical values with the mean.
* Median imputation: replaces missing numerical values with the median.
* Most-frequent imputation: replaces missing values with the most frequent value.

In scikit-learn, `SimpleImputer` performs these operations.

### B. Feature Scaling

Scaling adjusts the scale of numerical features.

For example:

* Tenure may range from 1 to 72 months.
* Monthly charges may range from 20 to 150.

`StandardScaler` standardizes numerical features using the mean and standard deviation learned from the training data.

Scaling is particularly useful for models such as Logistic Regression, K-Nearest Neighbors, and Support Vector Machines.

### C. Categorical Encoding

Encoding converts categorical values into a numerical representation.

For example:

| Contract Type  | Encoded Example |
| -------------- | --------------: |
| Month-to-month |               0 |
| One-year       |               1 |
| Two-year       |               2 |

The appropriate encoding depends on whether the categories are nominal or ordinal. For nominal categories, `OneHotEncoder` is often suitable.

### D. Machine Learning Model

The final pipeline component is the estimator that learns patterns from the prepared data.

Examples:

* Logistic Regression
* Decision Tree
* Random Forest
* Support Vector Machine

## 5. What Is `Pipeline` in Scikit-Learn?

`Pipeline` combines transformations and an estimator into a single object.

Example:

```python
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])
```

Each step has a name and an associated transformer or model.

When we call `pipeline.fit(X_train, y_train)`:

1. The imputer learns the median from the training features and fills missing values.
2. The scaler learns the training data's scaling parameters and scales the features.
3. Logistic Regression learns patterns from the processed training data.

When we call `pipeline.predict(X_test)`, the fitted preprocessing steps are applied to the test features before the model generates predictions.

## 6. What Is `ColumnTransformer`?

Real-world datasets often contain both numerical and categorical features.

Numerical features may need imputation and scaling, while categorical features may need imputation and encoding.

`ColumnTransformer` allows us to apply different preprocessing operations to different columns.

Example:

```python
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, ["Tenure", "MonthlyCharges"]),
    ("categorical", categorical_pipeline, ["ContractType"])
])
```

In this example:

* Numerical columns are imputed and scaled.
* Categorical columns are imputed and one-hot encoded.
* `handle_unknown="ignore"` helps the encoder handle categories that were not present during fitting.

A final model can be added after the `preprocessor` in an outer `Pipeline`.

## 7. How Does a Pipeline Prevent Data Leakage?

A pipeline helps prevent preprocessing leakage when it is fitted only on training data.

For example, if we use a pipeline with an imputer and scaler:

* The imputer learns its replacement values from training data.
* The scaler learns its parameters from training data.
* Test data is transformed using those already-fitted steps.

**Correct approach:**

1. Split the dataset.
2. Fit the pipeline on the training set.
3. Predict using the test set.
4. Evaluate the predictions.

**Incorrect approach:** Fit preprocessing steps on the complete dataset before splitting it.

Important: A pipeline cannot automatically prevent every type of leakage. We must still remove invalid target-derived features, respect time, and avoid using test results to tune the model.

## 8. Pipeline vs Individual Preprocessing Steps

| Individual Steps                                    | Pipeline                                                  |
| --------------------------------------------------- | --------------------------------------------------------- |
| Each step is managed separately                     | Steps are organized in one object                         |
| More repetitive code                                | Less repetitive code                                      |
| Easier to apply transformations inconsistently      | Applies fitted transformations consistently               |
| Leakage prevention depends on how each step is used | Helps prevent preprocessing leakage when correctly fitted |
| Can require more manual management                  | Easier to integrate with cross-validation                 |

## 9. Key Takeaways

* A preprocessing pipeline organizes data preparation and model training.
* `SimpleImputer` handles missing values.
* `StandardScaler` scales numerical features.
* `OneHotEncoder` converts categorical features into numerical columns.
* `ColumnTransformer` applies different transformations to different columns.
* `Pipeline` combines transformations and a final estimator.
* Fit the pipeline on training data only.
* Use the same fitted pipeline to predict on new data.
* Pipelines improve consistency and help prevent preprocessing leakage.


# Demonstration of the Concept – Preprocessing Pipeline

## Real-World Example: Customer Churn Prediction

Imagine a telecommunications company wants to predict whether a customer will leave its service.

The dataset contains these columns:

| Feature         | Data Type   | Possible Problem                                   | Preprocessing Step                            |
| --------------- | ----------- | -------------------------------------------------- | --------------------------------------------- |
| Tenure          | Numerical   | Missing values and a different scale               | Median imputation and scaling                 |
| Monthly Charges | Numerical   | Missing values and a different scale               | Median imputation and scaling                 |
| Contract Type   | Categorical | Text values cannot be directly used by many models | Most-frequent imputation and one-hot encoding |
| Churn           | Target      | Represents whether the customer leaves             | Used as the target variable                   |

The goal is to prepare the features and train a model without manually repeating each step.

## Step 1: Separate the Features and Target

The input features (`X`) contain tenure, monthly charges, and contract type.

The target (`y`) contains the churn label:

* `0` means the customer stayed.
* `1` means the customer left.

## Step 2: Split the Dataset

Divide the dataset into training and testing sets.

The training set is used to learn the preprocessing parameters and train the model. The test set is kept separate for final evaluation.

## Step 3: Preprocess Numerical Features

The numerical pipeline:

1. Fills missing values using the median.
2. Standardizes the numerical features.

## Step 4: Preprocess Categorical Features

The categorical pipeline:

1. Fills missing values using the most frequent category.
2. Uses one-hot encoding to convert categories into numerical columns.

## Step 5: Combine the Preprocessing Steps

`ColumnTransformer` applies the appropriate preprocessing to each group of columns.

An outer `Pipeline` combines the preprocessor with Logistic Regression.

## Step 6: Train and Evaluate the Model

Calling `pipeline.fit(X_train, y_train)` fits the preprocessing steps and model using training data.

Calling `pipeline.predict(X_test)` applies the fitted transformations and predicts the test labels.

We then compare the predictions with the actual test labels using evaluation metrics such as accuracy, precision, recall, and F1-score.

## Why Is This Useful?

Imagine receiving 100 new customer records tomorrow.

Without a pipeline, we would need to remember to apply the same imputation, scaling, and encoding operations in exactly the same way.

With a fitted pipeline, we can pass the new records to `pipeline.predict()` using the same input columns, and the pipeline applies its learned transformations automatically.

**Conclusion:** A preprocessing pipeline creates a repeatable and organized workflow for preparing data and training a Machine Learning model.


In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# --------------------------------------------------
# 1. Create a sample customer churn dataset
# --------------------------------------------------

data = pd.DataFrame({
    "Tenure": [
        2, 5, 8, 10, 12, 15, 18, 20, 22, 25,
        28, 30, 32, 35, 38, 40, 42, 45, 48, 50,
        3, 6, 9, 11, 14, 16, 19, 21, 24, 27,
        29, 31, 34, 36, 39, 41, 44, 46, 49, 52
    ],
    "MonthlyCharges": [
        90, 85, 80, 75, 70, 65, 60, 55, 50, 48,
        45, 42, 40, 38, 35, 33, 30, 28, 25, 22,
        88, 82, 78, 73, 68, 63, 58, 53, 49, 46,
        43, 41, 39, 36, 34, 31, 29, 27, 24, 20
    ],
    "ContractType": [
        "Monthly", "Monthly", "Monthly", "Monthly",
        "Yearly", "Yearly", "Monthly", "Yearly",
        "Two-Year", "Yearly", "Two-Year", "Yearly",
        "Two-Year", "Yearly", "Two-Year", "Yearly",
        "Two-Year", "Yearly", "Two-Year", "Two-Year",
        "Monthly", "Monthly", "Monthly", "Monthly",
        "Yearly", "Monthly", "Yearly", "Two-Year",
        "Monthly", "Yearly", "Two-Year", "Yearly",
        "Two-Year", "Yearly", "Two-Year", "Yearly",
        "Two-Year", "Yearly", "Two-Year", "Two-Year"
    ],
    "Churn": [
        1, 1, 1, 1, 1, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 1, 1, 1, 1, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0
    ]
})

# Introduce some missing values for demonstration
data.loc[2, "MonthlyCharges"] = float("nan")
data.loc[7, "Tenure"] = float("nan")
data.loc[15, "ContractType"] = None

print("Original Dataset:")
print(data.head(10))

print("\nMissing Values Before Preprocessing:")
print(data.isnull().sum())

# --------------------------------------------------
# 2. Separate features and target
# --------------------------------------------------

X = data.drop(columns=["Churn"])
y = data["Churn"]

# Identify numerical and categorical columns
numeric_features = ["Tenure", "MonthlyCharges"]
categorical_features = ["ContractType"]

# --------------------------------------------------
# 3. Split the dataset before preprocessing
# --------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("\nTraining Records:", len(X_train))
print("Testing Records:", len(X_test))

# --------------------------------------------------
# 4. Create the numerical preprocessing pipeline
# --------------------------------------------------

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# --------------------------------------------------
# 5. Create the categorical preprocessing pipeline
# --------------------------------------------------

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# --------------------------------------------------
# 6. Combine the preprocessing pipelines
# --------------------------------------------------

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# --------------------------------------------------
# 7. Create the complete Machine Learning pipeline
# --------------------------------------------------

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

# --------------------------------------------------
# 8. Train the complete pipeline
# --------------------------------------------------

pipeline.fit(X_train, y_train)

print("\nPipeline training completed successfully!")

# --------------------------------------------------
# 9. Make predictions on the test set
# --------------------------------------------------

y_pred = pipeline.predict(X_test)

print("\nActual Test Labels:")
print(y_test.to_list())

print("\nPredicted Test Labels:")
print(y_pred.tolist())

# --------------------------------------------------
# 10. Evaluate the model
# --------------------------------------------------

accuracy = accuracy_score(y_test, y_pred)

print("\nModel Evaluation")
print("----------------")
print("Test Accuracy:", round(accuracy, 3))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    zero_division=0
))

# --------------------------------------------------
# 11. Predict on new customer records
# --------------------------------------------------

new_customers = pd.DataFrame({
    "Tenure": [6, 45],
    "MonthlyCharges": [82.0, 30.0],
    "ContractType": ["Monthly", "Two-Year"]
})

new_predictions = pipeline.predict(new_customers)

print("\nPredictions for New Customers:")
print(new_customers.assign(PredictedChurn=new_predictions))

print("\nPreprocessing pipeline demonstration completed!")

Original Dataset:
   Tenure  MonthlyCharges ContractType  Churn
0     2.0            90.0      Monthly      1
1     5.0            85.0      Monthly      1
2     8.0             NaN      Monthly      1
3    10.0            75.0      Monthly      1
4    12.0            70.0       Yearly      1
5    15.0            65.0       Yearly      0
6    18.0            60.0      Monthly      0
7     NaN            55.0       Yearly      0
8    22.0            50.0     Two-Year      0
9    25.0            48.0       Yearly      0

Missing Values Before Preprocessing:
Tenure            1
MonthlyCharges    1
ContractType      1
Churn             0
dtype: int64

Training Records: 30
Testing Records: 10

Pipeline training completed successfully!

Actual Test Labels:
[0, 0, 0, 0, 0, 0, 0, 1, 1, 0]

Predicted Test Labels:
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Model Evaluation
----------------
Test Accuracy: 0.8

Classification Report:
              precision    recall  f1-score   support

           0       0